# Lab 2 Procesamiento de Lenguaje Natural

In [20]:
#!pip install sentence-transformers numpy scikit-learn

In [21]:
import re
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

#
  

In [22]:
oraciones = [
    # Instrumentos
    "La guitarra eléctrica es muy utilizada en la música rock.",
    "El piano permite interpretar melodías y acordes al mismo tiempo.",
    "La batería establece el ritmo principal de una canción.",
    "El violín es un instrumento de cuerda que se toca con un arco.",

    # Géneros musicales
    "El rock se caracteriza por el uso de guitarras, bajo y batería.",
    "El jazz utiliza frecuentemente la improvisación musical.",
    "La música clásica incluye obras de compositores como Mozart y Beethoven.",
    "El reguetón combina ritmos urbanos con sonidos electrónicos.",

    # Canto y voz
    "Un cantante debe controlar su respiración para mejorar su interpretación.",
    "La afinación permite cantar correctamente las notas musicales.",
    "El coro está formado por varias personas que cantan juntas.",
    "La voz puede clasificarse según su registro y tonalidad.",

    # Teoría musical
    "La melodía es una sucesión de notas que forma una idea musical.",
    "El ritmo organiza la duración de los sonidos y los silencios.",
    "La armonía combina diferentes notas que suenan simultáneamente.",
    "El tempo indica la velocidad con la que debe interpretarse una canción.",

    # Producción musical
    "Los productores musicales utilizan programas para editar grabaciones.",
    "Una canción puede grabarse utilizando micrófonos y una computadora.",
    "La mezcla permite equilibrar el volumen de los instrumentos y las voces.",
    "La masterización es la etapa final de producción de una canción.",
]

print(f"Total de oraciones en el corpus: {len(oraciones)}")

Total de oraciones en el corpus: 20


## Embeddings

In [23]:
# Se carga el modelo multilingüe de forma local (se descarga una sola vez)
modelo = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

# Generación de embeddings para todo el corpus
embeddings = modelo.encode(oraciones, show_progress_bar=True)

print(f"Shape de los embeddings: {embeddings.shape}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Shape de los embeddings: (20, 384)


### Función de búsqueda semántica (top-k coseno)

In [24]:
def buscar_semantica(consulta, top_k=3):
    # Busca las oraciones más similares a la consulta usando
    # similitud coseno sobre embeddings locales.

    embedding_consulta = modelo.encode([consulta])
    similitudes = cosine_similarity(embedding_consulta, embeddings)[0]
    indices_top = np.argsort(similitudes)[::-1][:top_k]

    resultados = []
    for idx in indices_top:
        resultados.append({
            "oracion": oraciones[idx],
            "similitud": round(float(similitudes[idx]), 4)
        })
    return resultados


def mostrar_resultados_semanticos(consulta, top_k=3):

    print(f"\nConsulta semántica: '{consulta}'")
    resultados = buscar_semantica(consulta, top_k)
    for i, r in enumerate(resultados, 1):
        print(f"  {i}. [{r['similitud']:.4f}] {r['oracion']}")

### Función de búsqueda por palabras clave (TF simple)

In [25]:
def tokenizar(texto):
    #Convierte texto a lista de tokens en minúsculas
    texto = texto.lower()
    return set(re.findall(r"\b\w+\b", texto, flags=re.UNICODE))


def buscar_keywords(consulta, top_k=3):

    # Busca oraciones que compartan más palabras clave con la consulta.
    # Score = cantidad de tokens en común (sin stopwords avanzadas).
    tokens_consulta = tokenizar(consulta)
    scores = []

    for oracion in oraciones:
        tokens_oracion = tokenizar(oracion)
        coincidencias = tokens_consulta & tokens_oracion
        scores.append(len(coincidencias))

    indices_top = np.argsort(scores)[::-1][:top_k]

    resultados = []
    for idx in indices_top:
        resultados.append({
            "oracion": oraciones[idx],
            "coincidencias": scores[idx]
        })
    return resultados


def mostrar_resultados_keywords(consulta, top_k=3):

    print(f"\nBúsqueda por keywords: '{consulta}'")

    resultados = buscar_keywords(consulta, top_k)
    for i, r in enumerate(resultados, 1):
        print(f"  {i}. [coincidencias: {r['coincidencias']}] {r['oracion']}")

In [26]:
consultas = [
    "¿Qué instrumentos se usan en el rock?",
    "¿Cómo se graba una canción en casa?",
    "quiero aprender a cantar mejor",
    "diferencia entre melodía y armonía",
    "músicos que improvisan en el jazz",
]

for consulta in consultas:
    mostrar_resultados_semanticos(consulta, top_k=3)
    mostrar_resultados_keywords(consulta, top_k=3)
    print()


Consulta semántica: '¿Qué instrumentos se usan en el rock?'
  1. [0.8352] El rock se caracteriza por el uso de guitarras, bajo y batería.
  2. [0.7371] La guitarra eléctrica es muy utilizada en la música rock.
  3. [0.4963] El piano permite interpretar melodías y acordes al mismo tiempo.

Búsqueda por keywords: '¿Qué instrumentos se usan en el rock?'
  1. [coincidencias: 3] El rock se caracteriza por el uso de guitarras, bajo y batería.
  2. [coincidencias: 2] La mezcla permite equilibrar el volumen de los instrumentos y las voces.
  3. [coincidencias: 2] El violín es un instrumento de cuerda que se toca con un arco.


Consulta semántica: '¿Cómo se graba una canción en casa?'
  1. [0.6415] Una canción puede grabarse utilizando micrófonos y una computadora.
  2. [0.4770] Los productores musicales utilizan programas para editar grabaciones.
  3. [0.4171] La afinación permite cantar correctamente las notas musicales.

Búsqueda por keywords: '¿Cómo se graba una canción en casa?'
  1. [coi